In [ ]:

# ============ REGRESSION: PCA vs WITHOUT PCA ============

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from scipy.stats import ttest_rel

from sklearn.model_selection import (
    train_test_split, GridSearchCV, KFold, cross_validate
)
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.decomposition import PCA
from sklearn.svm import SVR
from sklearn.metrics import (
    mean_squared_error, mean_absolute_error, r2_score,
    mean_absolute_percentage_error
)

# ================= 1. LOAD DATASET =================
df = pd.read_csv("your_dataset.csv")
target = "target"                    # Change target column
df = df.drop_duplicates()
df = df.drop(columns=["id"], errors="ignore")

# ================= 2. EDA =================
print(df.info())
print(df.describe(include="all"))
print("\nMissing values:\n", df.isnull().sum())

df.select_dtypes(include=np.number).hist(figsize=(10, 6))
plt.tight_layout()
plt.show()

sns.heatmap(df.select_dtypes(include=np.number).corr(),
            cmap="coolwarm")
plt.show()

sns.histplot(data=df, x=target, kde=True)
plt.show()

# ================= 3. FEATURES & TARGET =================
X = df.drop(columns=target)
y = df[target]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

num = X.select_dtypes(include=np.number).columns.tolist()
cat = X.select_dtypes(exclude=np.number).columns.tolist()

# ================= 4. PREPROCESSING =================
prep = ColumnTransformer([
    ("num", Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler())
    ]), num),

    ("cat", Pipeline([
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("encoder", OneHotEncoder(handle_unknown="ignore",
                                  sparse_output=False))
    ]), cat)
])

# ================= 5. MODELS =================
pipe1 = Pipeline([
    ("prep", prep),
    ("model", SVR(kernel="rbf"))
])

pipe2 = Pipeline([
    ("prep", prep),
    ("pca", PCA(n_components=0.95)),
    ("model", SVR(kernel="rbf"))
])

params = {
    "model__C": [1, 10, 100],
    "model__gamma": ["scale", 0.1],
    "model__epsilon": [0.1, 0.2]
}

cv = KFold(n_splits=5, shuffle=True, random_state=42)

# ================= 6. HYPERPARAMETER TUNING =================
grid1 = GridSearchCV(
    pipe1, params, cv=cv,
    scoring="neg_root_mean_squared_error", n_jobs=-1
)

grid2 = GridSearchCV(
    pipe2, params, cv=cv,
    scoring="neg_root_mean_squared_error", n_jobs=-1
)

grid1.fit(X_train, y_train)
grid2.fit(X_train, y_train)

print("\nWithout PCA best parameters:", grid1.best_params_)
print("Without PCA CV RMSE:", -grid1.best_score_)

print("\nWith PCA best parameters:", grid2.best_params_)
print("With PCA CV RMSE:", -grid2.best_score_)

# ================= 7. PERFORMANCE METRICS =================
for name, model in [
    ("Without PCA", grid1.best_estimator_),
    ("With PCA", grid2.best_estimator_)
]:
    pred = model.predict(X_test)

    print("\n", name)
    print("RMSE:", np.sqrt(mean_squared_error(y_test, pred)))
    print("MAE:", mean_absolute_error(y_test, pred))
    print("R2:", r2_score(y_test, pred))

    if np.all(y_test != 0):
        print("MAPE (%):",
              mean_absolute_percentage_error(y_test, pred) * 100)

    if "pca" in model.named_steps:
        pca = model.named_steps["pca"]
        print("PCA components:", pca.n_components_)
        print("Explained variance:", pca.explained_variance_ratio_.sum())

# ================= 8. CROSS-VALIDATION SCORES =================
scores1 = cross_validate(
    grid1.best_estimator_, X_train, y_train,
    cv=cv, scoring=["neg_root_mean_squared_error",
                    "neg_mean_absolute_error", "r2"]
)

scores2 = cross_validate(
    grid2.best_estimator_, X_train, y_train,
    cv=cv, scoring=["neg_root_mean_squared_error",
                    "neg_mean_absolute_error", "r2"]
)

rmse1 = -scores1["test_neg_root_mean_squared_error"]
rmse2 = -scores2["test_neg_root_mean_squared_error"]

mae1 = -scores1["test_neg_mean_absolute_error"]
mae2 = -scores2["test_neg_mean_absolute_error"]

print("\nMean CV RMSE without PCA:", rmse1.mean())
print("Mean CV RMSE with PCA:", rmse2.mean())

print("Mean CV MAE without PCA:", mae1.mean())
print("Mean CV MAE with PCA:", mae2.mean())

print("Mean CV R2 without PCA:", scores1["test_r2"].mean())
print("Mean CV R2 with PCA:", scores2["test_r2"].mean())

# ================= 9. PAIRED T-TEST =================
t_stat, p_value = ttest_rel(rmse1, rmse2)

print("\nRMSE paired t-test")
print("t-statistic:", t_stat)
print("p-value:", p_value)

if p_value < 0.05:
    print("Significant difference in RMSE")
else:
    print("No significant difference in RMSE")

t_stat, p_value = ttest_rel(mae1, mae2)

print("\nMAE paired t-test")
print("t-statistic:", t_stat)
print("p-value:", p_value)

if p_value < 0.05:
    print("Significant difference in MAE")
else:
    print("No significant difference in MAE")
